# Medical Image Triage with vision-language models and vLLM

This notebook provides a step-by-step guide to building a medical image triage system
using a vision-language model with vLLM and Gradio for inference.
 
The goal is not to make a definitive diagnosis. Instead, the model should classify the image
into a conservative triage category, identify whether it looks like an X-ray, a normal photo, or a prescription,
and extract prescription text when present.
 
This tutorial explores how to use a vision model for first-pass triage and then hand the structured output
to a Qwen-based conversational layer in the companion notebook. It covers the following topics:

[Installing dependencies](#install-deps)  
[Building medical image triage with vLLM](#cli-ocr)  
[Transforming triage into a GUI-enabled system with multiple model choices](#gradio-gui)


The tutorial uses vLLM for large language model (LLM) inference. vLLM optimizes text generation workloads by effectively batching requests and utilizing GPU resources, offering high throughput and low latency.

![Medical Triage Example](./assets/ocr.gif)
---

## Prerequisites

### Hugging Face API access

* Obtain an API token from [Hugging Face](https://huggingface.co) for downloading models.
* Ensure the Hugging Face API token has the necessary permissions and approval to access the [Meta Llama checkpoints](https://huggingface.co/meta-llama/Llama-3.1-8B).

<a id="install-deps"></a>
## 1. Installing dependencies

Install the libraries needed for this tutorial. Run the following commands inside the Jupyter notebook running within the Docker container:

In [ ]:
!pip install gradio requests

### Provide your Hugging Face token

You'll require a Hugging Face API token to access Llama-3.1-8B. Generate your token at [Hugging Face Tokens](https://huggingface.co/settings/tokens) and request access for [Llama-3.2-11B-Vision-Instruct](https://huggingface.co/meta-llama/Llama-3.2-11B-Vision-Instruct). Tokens typically start with "hf_".

Run the following interactive block in your Jupyter notebook to set up the token:

In [ ]:
from huggingface_hub import notebook_login, HfApi

# Prompt the user to log in
notebook_login()


Verify that your token was accepted correctly:

In [ ]:
# Validate the token
try:
    api = HfApi()
    user_info = api.whoami()
    print(f"Token validated successfully! Logged in as: {user_info['name']}")
except Exception as e:
    print(f"Token validation failed. Error: {e}")


<a id="cli-ocr"></a>
## 2. Building medical image triage with vLLM

First, define the `ImageInference` inference class. This class provides a constructor to initialize a model for inference. Additionally, it defines another function that runs triage inference on an image that you provide. 

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from PIL import Image

current_model = "meta-llama/Llama-3.2-11B-Vision-Instruct"

class ImageInference:
    def __init__(self, model_name=current_model):
        self.model_name = model_name
        self.llm = LLM(model=model_name, max_model_len=4096, max_num_seqs=16, enforce_eager=True)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    def generate_image_output(self, image: Image, patient_context: str = "") -> str:
        context_block = f"Patient context: {patient_context}\n" if patient_context else ""
        messages = [{
            'role': 'user',
            'content': (
                "You are a medical image triage assistant. Analyze the provided <|image|> image and return a concise structured assessment.\n"
                "Classify the image as one of: xray, normal_photo, prescription, or unknown.\n"
                "If the image looks like a prescription, extract the visible text exactly.\n"
                "If the image looks like a medical photo or X-ray, give a conservative triage label such as normal, monitor, urgent, or emergency.\n"
                "Use the following format exactly:\n"
                "image_type: <one of xray|normal_photo|prescription|unknown>\n"
                "triage_label: <normal|monitor|urgent|emergency|not_applicable>\n"
                "summary: <one short sentence>\n"
                "findings: <bullet-style semicolon-separated details>\n"
                "prescription_text: <exact text or none>\n"
                "follow_up_questions: <up to 3 questions for the doctor-style Q&A layer>\n"
                f"{context_block}"
                "Do not provide a final diagnosis. Do not add commentary outside the requested format."
            )
        }]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        sampling_params = SamplingParams(max_tokens=512, temperature=0.2)

        outputs = self.llm.generate({
            "prompt": prompt,
            "multi_modal_data": {"image": image},
        }, sampling_params=sampling_params)

        generated_text = outputs[0].outputs[0].text if outputs else "No output generated."
        return generated_text

### Testing the triage system

Download [this test image](https://github.com/ROCm/gpuaidev/tree/main/docs/notebooks/assets/together_we_advance_.png) by running the following command:

In [ ]:
import requests

url = "https://raw.githubusercontent.com/ROCm/gpuaidev/main/docs/notebooks/assets/together_we_advance_.png"
filename = "together_we_advance_.png"

response = requests.get(url)
with open(filename, "wb") as file:
    file.write(response.content)

print("Download complete:", filename)

Now it's time to test your triage system. First read the image, then view it.

In [ ]:
pil_image = Image.open("together_we_advance_.png")
pil_image = pil_image.convert("RGB")  # Ensure the image is in RGB format
pil_image

Next, initialize an instance of the `ImageInference` class.

In [ ]:
# Initialize the inference class
inference = ImageInference()

Now pass the image to the model for inference and print the results.

In [ ]:

# Generate output for the image
output = inference.generate_image_output(pil_image)

# Print the result
print("Model Output:")
print(output)

Congratulations. You've just built a medical image triage system. That gives you a first-pass classifier and a structured handoff for a doctor-style Q&A flow.

<a id="gradio-gui"></a>
## 3. Transforming triage into a GUI-enabled system with multiple model choices

To provide a graphical interface for your assistant, use [Gradio](https://www.gradio.app/) to create an interactive web-based UI.

### Import Gradio and define a shortlist of vision models to access

You're going to create a shortlist of models that can analyze medical images. The full list is available [here](https://docs.vllm.ai/en/latest/models/supported_models.html).

In [ ]:
import gradio as gr

# Define available models and their Hugging Face model IDs
MODEL_OPTIONS = {
    "Llama-3.2-11B-Vision-Instruct": "meta-llama/Llama-3.2-11B-Vision-Instruct",
    "Qwen2-VL (2B)": "Qwen/Qwen2-VL-2B-Instruct",
    "Qwen2-VL (7B)": "Qwen/Qwen2-VL-7B-Instruct",
    "Phi-3.5 Vision": "microsoft/Phi-3.5-vision-instruct",
}

In [ ]:
import hashlib
import json
import os
from typing import Any

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

VECTOR_DB_PATH = os.path.join(os.getcwd(), "medical_memory_chroma")
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

embedding_function = SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL)
chroma_client = chromadb.PersistentClient(path=VECTOR_DB_PATH)
medical_collection = chroma_client.get_or_create_collection(
    name="medical_triage_notes",
    embedding_function=embedding_function,
)


def triage_text_to_dict(text: str) -> dict[str, Any]:
    out = {}
    for line in text.splitlines():
        line = line.strip()
        if not line or ':' not in line:
            continue
        k, v = line.split(':', 1)
        out[k.strip()] = v.strip()
    if 'follow_up_questions' in out:
        out['follow_up_questions'] = [q.strip() for q in out['follow_up_questions'].split(',') if q.strip()]
    return out


def save_triage_json(triage_report: dict[str, Any], path: str = "triage_report.json") -> str:
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(triage_report, f, ensure_ascii=False, indent=2)
    return path


def upsert_triage_to_chroma(triage_report: dict[str, Any], conversation_id: str = "default") -> str:
    document = "\n".join(
        [
            f"image_type: {triage_report.get('image_type', '')}",
            f"triage_label: {triage_report.get('triage_label', '')}",
            f"summary: {triage_report.get('summary', '')}",
            f"findings: {triage_report.get('findings', '')}",
            f"prescription_text: {triage_report.get('prescription_text', 'none')}",
            f"follow_up_questions: {', '.join(triage_report.get('follow_up_questions', []))}",
        ]
    )
    record_id = hashlib.sha1(f"{conversation_id}:{document}".encode('utf-8')).hexdigest()
    medical_collection.upsert(
        ids=[record_id],
        documents=[document],
        metadatas=[{"conversation_id": conversation_id, "kind": "triage_report"}],
    )
    return record_id


In [ ]:
!pip install -q chromadb sentence-transformers


### Create the Gradio interface

Next, define two simple helper functions for switching models and running inference using your previously defined class, followed by Gradio blocks:

In [ ]:
def switch_model(model_name):
    global inference, current_model
    try:
        inference = ImageInference(model_name=model_name)
        current_model = model_name
        return f"Switched to model: {model_name}", f"Current Model: {current_model}"
    except Exception as e:
        return f"Failed to switch model: {str(e)}", f"Current Model: {current_model}"


def analyze_image(image, patient_context):
    try:
        if image is None:
            return "Please upload an image first."
        pil_image = image.convert("RGB")
        result_text = inference.generate_image_output(pil_image, patient_context=patient_context or "")
        # Parse generated triage text into a dict
        triage_report = triage_text_to_dict(result_text)
        # Save JSON handoff
        try:
            json_path = save_triage_json(triage_report)
        except Exception:
            json_path = None
        # Upsert into Chroma vector DB
        try:
            record_id = upsert_triage_to_chroma(triage_report)
        except Exception:
            record_id = None
        output_lines = ["### Model Output", "", "```", result_text, "```", ""]
        if json_path:
            output_lines += [f"Saved triage JSON: {json_path}"]
        if record_id:
            output_lines += [f"Stored triage record id: {record_id}"]
        return "\n".join(output_lines)
    except Exception as e:
        return f"Error processing the image: {str(e)}"


Execute the code block below to launch the GUI. The interface displays in your browser, letting you interact with the triage system.

In [ ]:
interface.launch(share=True)

## Conclusion

In this tutorial, you accomplished the following tasks:

* Built a medical image triage class using vLLM.
* Extended the functionality by adding a GUI and a selection of multiple different models.
* Produced a structured handoff that can feed the Qwen follow-up notebook.

Happy coding! If you encounter issues or have questions, don’t hesitate to ask or raise an issue on our [Github page](https://github.com/ROCm/gpuaidev)!